In [1]:
def intrinsic_efficiency(detector, dist=0.1):   
    #Whatever the intrinisic effeciencie ends up being you can append it to the result in absolute effciency
    """
    Function that calculates the intrinsic efficiency of the detector.
    This is done by determining and applying a geometry factor which 
    itself is dependant on the detectors radius and distance from 
    the radio isotope source.
    __________________
    Parameters
        dist: Float
        The distance the detector is from the radioisotope

        radius: Float
        radius of the detector

        abs_effic: Float
        absolute efficiency of the detector
    ___________________
    Outputs
        intrinsic_efficiency: Float
        intrinsic efficiency of the detector
    """
    rad_list = {
        "BGO": 0.0254,
        "NaI": 0.02921,
        "CdTe": 0.005,
    }

    intrinsic_results = {}

    radius = rad_list[detector.name]

    # calculating geometry factor
    G = 0.5 * (1 - dist / np.sqrt(dist**2 + radius**2))

    for source, peaks in detector.efficiency.items():

        intrinsic_results[source] = []   
        
        for peak in peaks:
            abs_eff = peak["absolute_efficiency"]
            abs_err = peak["efficiency_error"]
            intrinsic_efficiency = abs_eff / G
            intrinsic_error = abs_err / G
            energy = peak["energy"]

            result = {
                "source": source,
                "energy": energy,
                "intrinsic_efficiency": intrinsic_efficiency,
                "intrinsic_error": intrinsic_error,
            }

            intrinsic_results[source].append(result)

            print(
                f"\n  Energy: {energy:.3f} keV"
                f"\n  Intrinsic efficiency: "
                f"{intrinsic_efficiency:.6e} +/- "
                f"{intrinsic_error:.2e}"
            )

    detector.intrinsic_efficiency = intrinsic_results

    print("\n" + "=" * 65)
    print("INTRINSIC EFFICIENCY CALCULATION COMPLETE")
    print("=" * 65)

    return intrinsic_results



def plot_intrinsic_efficiency(detector):
    """
    Plot absolute efficiency against gamma-ray energy
    for all sources belonging to a detector.
    """

    if detector.intrinsic_efficiency is None:
        raise ValueError("Calculate efficiency first.")

    plt.figure(figsize=(8, 5))

    for source, peaks in detector.intrinsic_efficiency.items():

        if not peaks:
            continue

        energies = [p["energy"] for p in peaks]
        efficiencies = [
            p["intrinsic_efficiency"] for p in peaks
        ]
        errors = [
            p["intrinsic_error"] for p in peaks
        ]

        plt.errorbar(
            energies,
            efficiencies,
            yerr=errors,
            fmt="o",
            capsize=4,
            label=source
        )

    plt.xlabel("Gamma-ray Energy (keV)")
    plt.ylabel("Intrinsic Efficiency")
    plt.title(f"{detector.name} Intrinsic Efficiency")
    plt.grid(alpha=0.3)
    plt.legend(title="Source")
    plt.tight_layout()
    plt.show()